In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

def add_year_month(da):
    da = da.squeeze(drop=True)
    if "member_id" in da.dims:
        da = da.isel(member_id=0, drop=True)
    if "dcpp_init_year" in da.dims:
        da = da.isel(dcpp_init_year=0, drop=True)
    years = da["time"].dt.year.data
    months = da["time"].dt.month.data
    return da.assign_coords(year=("time", years), month=("time", months))

model = "NorESM2-MM"#ACCESS-CM2 BCC-CSM2-MR
era5 = xr.open_dataset(Path("../Datos_Era5/era5_guajira_daily_1981_2014.nc"))["wind10"]
cmip = xr.open_dataset(Path(f"./corrected/{model}_historical_wind_bc.nc"))["wind_bc"]

era5_ym = add_year_month(era5)
cmip_ym = add_year_month(cmip)

common_years = np.intersect1d(era5_ym["year"], cmip_ym["year"])
era5_ym = era5_ym.where(era5_ym["year"].isin(common_years), drop=True)
cmip_ym = cmip_ym.where(cmip_ym["year"].isin(common_years), drop=True)

era5_m = era5_ym.groupby("month").mean("time").to_series()
cmip_m = cmip_ym.groupby("month").mean("time").to_series()

months = pd.Index(range(1, 13), name="month")
df = pd.DataFrame(
    {"ERA5_wind10": era5_m.reindex(months), "CMIP_wind_bc": cmip_m.reindex(months)}
)
display(df)


c:\Users\LINDA CATALINA\AppData\Local\Programs\Python\Python312\Lib\site-packages\xarray\backends\plugins.py:109: RuntimeWarning: Engine 'gini' loading failed:
No module named 'pyproj'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


,ERA5_wind10,CMIP_wind_bc
month,,
1,7.839013,7.839012
2,8.141427,8.145494
3,7.849468,7.849468
4,7.299322,7.299322
5,6.864361,6.864361
6,8.240860,8.240861
7,8.565274,8.565275
8,7.124693,7.124693
9,5.309645,5.309645


##### Esa tabla es la climatología mensual de ERA5 vs el viento corregido (wind_bc) para el histórico de ese modelo. Los valores están alineados mes a mes y son prácticamente idénticos (difieren sólo en el redondeo), lo cual indica que la corrección media/σ se aplicó correctamente: la climatología mensual del modelo corregido coincide con la de ERA5 en el periodo histórico.

In [2]:
import xarray as xr
import pandas as pd
from pathlib import Path

model = "NorESM2-MM"  # o el que estés viendo
ds = xr.open_dataset(Path(f"./corrected/{model}_ssp585_wind_bc.nc"))
wind = ds["wind_bc"].squeeze(drop=True)
nan_per_month = wind.groupby("time.month").apply(lambda x: x.isnull().sum())
print(nan_per_month)


<xarray.DataArray 'wind_bc' (month: 12)> Size: 96B
array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Coordinates:
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
    height   float64 8B 10.0
    lat      float64 8B 11.78
    lon      float64 8B -72.5


In [3]:
import xarray as xr
import numpy as np
from pathlib import Path

model = "NorESM2-MM"#ACCESS-CM2 #CanESM5 salió nan
exp = "historical"  # o historical, ssp245, ssp585
ds = xr.open_dataset(Path(f"./corrected/{model}_{exp}_wind_bc.nc"))

cf_min = float(ds["cf"].min())
cf_max = float(ds["cf"].max())
print(f"cf min/max: {cf_min:.3f}, {cf_max:.3f}")

rho_ref = 1.225
rating = 6.8
losses = 0.10
if "rho" in ds:
    scale = (ds["rho"] / rho_ref) * (1 - losses)
    p_limit = rating * scale
    p_max_allowed = float(p_limit.max())
    p_max = float(ds["power_MW"].max())
    print(f"power_MW max: {p_max:.3f} MW; límite teórico máx (densidad/pérdidas): {p_max_allowed:.3f} MW")
else:
    print("rho no disponible; revisar power_MW <= 6.8*(1-0.1)")
    p_max = float(ds["power_MW"].max())
    print(f"power_MW max: {p_max:.3f} MW; límite nominal sin densidad: {6.8*(1-0.1):.3f} MW")


cf min/max: 0.000, 0.900
power_MW max: 6.120 MW; límite teórico máx (densidad/pérdidas): 6.120 MW


El “rho no disponible” en ssp585 para ACCESS-CM2 significa que no pudimos interpolar ps/tas en ese experimento, así que la potencia se calculó sin ajuste de densidad (se usó la curva nominal y pérdidas).

##### Significa que, para ese modelo/escenario:
* cf (factor de planta diario) está entre 0 y 0.855. Es menor a 1, lo cual es lógico; el máximo diario llegó al 85.5% de la potencia nominal.
* power_MW max es 5.811 MW, y coincide con el “límite teórico máx (densidad/pérdidas)” calculado como rating 6.8 MW × (rho/rho_ref) × (1–pérdidas). Esto indica que:
- La potencia no excede la curva esperada.
- El ajuste por densidad y la penalización de pérdidas están activos y coherentes.

In [4]:
import xarray as xr
ds = xr.open_dataset("corrected/NorESM2-MM_ssp585_wind_bc.nc")
print(ds)
for var in ["wind_bc", "v150", "power_MW", "cf"]:
    if var in ds:
        print(var, "time size:", ds[var].sizes.get("time", 0), "NaN:", int(ds[var].isnull().sum()))

<xarray.Dataset> Size: 2MB
Dimensions:         (member_id: 1, dcpp_init_year: 1, time: 31390)
Coordinates:
  * member_id       (member_id) <U8 32B 'r1i1p1f1'
  * dcpp_init_year  (dcpp_init_year) float64 8B nan
  * time            (time) object 251kB 2015-01-01 12:00:00 ... 2100-12-31 12...
    height          float64 8B ...
    lat             float64 8B ...
    lon             float64 8B ...
Data variables:
    wind_bc         (member_id, dcpp_init_year, time) float64 251kB ...
    v150            (member_id, dcpp_init_year, time) float64 251kB ...
    power_MW        (member_id, dcpp_init_year, time) float64 251kB ...
    cf              (member_id, dcpp_init_year, time) float64 251kB ...
    rho             (member_id, dcpp_init_year, time) float64 251kB ...
wind_bc time size: 31390 NaN: 0
v150 time size: 31390 NaN: 0
power_MW time size: 31390 NaN: 0
cf time size: 31390 NaN: 0


In [5]:
import xarray as xr
from pathlib import Path

model = "MIROC6"
for exp in ["historical", "ssp126"]:
    ds = xr.open_dataset(Path(f"./corrected/{model}_{exp}_wind_bc.nc"))
    print(f"\n{model} {exp}:")
    print(ds)
    for var in ["wind_bc", "v150", "power_MW", "cf"]:
        if var in ds:
            n_nan = int(ds[var].isnull().sum())
            print(f"  {var}: NaN={n_nan}, min={float(ds[var].min()):.3f}, max={float(ds[var].max()):.3f}")



MIROC6 historical:
<xarray.Dataset> Size: 497kB
Dimensions:         (member_id: 1, dcpp_init_year: 1, time: 12418)
Coordinates:
  * member_id       (member_id) <U8 32B 'r1i1p1f1'
  * dcpp_init_year  (dcpp_init_year) float64 8B nan
  * time            (time) datetime64[ns] 99kB 1981-01-01T12:00:00 ... 2014-1...
    height          float64 8B ...
    lat             float64 8B ...
    lon             float64 8B ...
Data variables:
    wind_bc         (member_id, dcpp_init_year, time) float64 99kB ...
    v150            (member_id, dcpp_init_year, time) float64 99kB ...
    power_MW        (member_id, dcpp_init_year, time) float64 99kB ...
    cf              (member_id, dcpp_init_year, time) float64 99kB ...
  wind_bc: NaN=0, min=-2.034, max=14.139
  v150: NaN=0, min=-2.972, max=20.657
  power_MW: NaN=0, min=0.000, max=6.120
  cf: NaN=0, min=0.000, max=0.900

MIROC6 ssp126:
<xarray.Dataset> Size: 2MB
Dimensions:         (member_id: 1, dcpp_init_year: 1, time: 31411)
Coordinates:
  * me

La curva de potencia que usamos pone potencia en 0 cuando el viento está por debajo de la velocidad de arranque (cut-in, 3 m/s). En esos días, la función devolvió NaN en vez de 0 para power_MW y cf en unos ~30 puntos. 
Así power_MW y cf serían exactamente 0 cuando el viento está por debajo de cut-in, sin NaN. El comportamiento físico es el mismo: en esos días la turbina no produce.

In [6]:
import xarray as xr
import numpy as np
from pathlib import Path

model = "NorESM2-MM"
exp = "historical"
ds = xr.open_dataset(Path(f"./corrected/{model}_{exp}_wind_bc.nc"))
dt = np.diff(ds["time"].values).astype("timedelta64[D]").astype(int)
print(f"Saltos únicos en días: {set(dt.tolist())}")


Saltos únicos en días: {1}


saltos de tiempo {1} indica serie diaria sin huecos.

In [7]:
import numpy as np

v_tab = np.array([3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0,
                  10.5, 11.0, 11.5, 12.0, 12.5, 13.0, 13.5, 14.0, 14.5, 15.0, 15.5, 16.0, 16.5,
                  17.0, 17.5, 18.0, 18.5, 19.0, 19.5, 20.0, 20.5, 21.0, 21.5, 22.0, 22.5, 23.0,
                  23.5, 24.0, 24.5, 25.0])
p_tab_kw = np.array([0, 40, 100, 370, 650, 895, 1150, 1500, 1850, 2375, 2900, 3525, 4150, 4875, 5600,
                     6350, 7100, 7580, 7800, 7920, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
                     8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
                     8000, 8000, 8000, 8000])
scale = 6.8 / 8.0
v_tab_comp = v_tab * (12/13); v_tab_comp[0] = 3.0
for vtest in [3, 6, 10, 12, 15]:
    print(vtest, np.interp(vtest, v_tab_comp, p_tab_kw/1000*scale))


3 0.0
6 1.275
10 5.822499999999998
12 6.8
15 6.8
